# CCS Empresa ABC SA — Valoración Detallada

Análisis completo del Cross-Currency Swap confirmado en el PDF **CCS 2026.02.16 EMPRESA ABC SA / Bancolombia**.

---

## Términos del Trade (del PDF)

| Campo | Valor |
|---|---|
| Contraparte A | **EMPRESA ABC SA** |
| Contraparte B | Bancolombia |
| Tipo | Cross-Currency Swap (CCS) amortizable |
| Nocional USD | **USD 1,000,000** |
| Nocional COP | **COP 3,661,000,000** |
| FX Inception | **3,661.00** |
| Inicio | **2026-02-17** |
| Vencimiento | **2028-02-17** |
| Frecuencia | Trimestral |
| Amortización | 8 pagos iguales de **12.5%** cada trimestre |
| Pata A — Empresa **paga** | SOFR 3M − 22 bps, ACT/360 |
| Pata B — Bancolombia **paga** | IBR trimestral flat, ACT/360 |

**Perspectiva:** Empresa paga USD (SOFR − 22bps) y recibe COP (IBR). `pay_usd = True`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'), override=True)

import QuantLib as ql
import pandas as pd
from datetime import datetime

from pricing.data.market_data import MarketDataLoader
from pricing.curves.curve_manager import CurveManager
from utilities.colombia_calendar import calendar_colombia

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

loader = MarketDataLoader()
print('MarketDataLoader OK')

MarketDataLoader OK


---
## 1. Constantes del Trade

In [2]:
# ── Trade constants (from PDF confirmation) ──────────────────────────────────
USD_NOTIONAL  = 1_000_000
FX_INCEPTION  = 3_661.00
COP_NOTIONAL  = USD_NOTIONAL * FX_INCEPTION   # 3,661,000,000
SOFR_SPREAD   = -22 / 10_000                  # -22 bps = -0.0022
AMORT_FACTOR  = 0.125                          # 12.5% each quarter
N_PERIODS     = 8
T_START       = ql.Date(17, 2, 2026)
T_MATURITY    = ql.Date(17, 2, 2028)
DC            = ql.Actual360()

# Notional at the START of each period (beginning-of-period balance)
# Period 1: 100%, Period 2: 87.5%, ..., Period 8: 12.5%
AMORT_USD = [USD_NOTIONAL * (1.0 - AMORT_FACTOR * i) for i in range(N_PERIODS)]
AMORT_COP = [n * FX_INCEPTION for n in AMORT_USD]

print(f'USD notional:   {USD_NOTIONAL:>15,.0f}')
print(f'FX inception:   {FX_INCEPTION:>15,.2f}')
print(f'COP notional:   {COP_NOTIONAL:>15,.0f}')
print(f'SOFR spread:    {SOFR_SPREAD*10000:>15.0f} bps')
print(f'Periods:        {N_PERIODS:>15}')
print()
print('  P   Amort%     USD notional     COP notional')
for i in range(N_PERIODS):
    pct = (1.0 - AMORT_FACTOR * i) * 100
    print(f'  {i+1}   {pct:5.1f}%   {AMORT_USD[i]:>12,.0f}   {AMORT_COP[i]:>18,.0f}')

USD notional:         1,000,000
FX inception:          3,661.00
COP notional:     3,661,000,000
SOFR spread:                -22 bps
Periods:                      8

  P   Amort%     USD notional     COP notional
  1   100.0%      1,000,000        3,661,000,000
  2    87.5%        875,000        3,203,375,000
  3    75.0%        750,000        2,745,750,000
  4    62.5%        625,000        2,288,125,000
  5    50.0%        500,000        1,830,500,000
  6    37.5%        375,000        1,372,875,000
  7    25.0%        250,000          915,250,000
  8    12.5%        125,000          457,625,000


---
## 2. Construcción del Schedule de Pagos

Calendario conjunto Colombia + FederalReserve (si alguno tiene festivo, se ajusta al siguiente día hábil `Following`).

In [3]:
cal_col = calendar_colombia()
cal_usd = ql.UnitedStates(ql.UnitedStates.FederalReserve)
cal_joint = ql.JointCalendar(cal_col, cal_usd)

schedule = ql.Schedule(
    T_START, T_MATURITY,
    ql.Period(3, ql.Months),
    cal_joint,
    ql.Following, ql.Following,
    ql.DateGeneration.Forward, False,
)
DATES = list(schedule)

print(f'Fechas en schedule: {len(DATES)}  →  {len(DATES)-1} períodos')
print()

rows = []
for i in range(1, len(DATES)):
    s, e = DATES[i-1], DATES[i]
    days = DC.dayCount(s, e)
    tau  = DC.yearFraction(s, e)
    rows.append({
        'P': i,
        'start':         str(s),
        'end':           str(e),
        'días (ACT)':    days,
        'tau (ACT/360)': round(tau, 6),
        'N_USD':         AMORT_USD[i-1],
        'N_COP':         AMORT_COP[i-1],
    })

sched_df = pd.DataFrame(rows)
sched_df

Fechas en schedule: 9  →  8 períodos



,P,start,end,días (ACT),tau (ACT/360),N_USD,N_COP
0,1,"February 17th, 2026","May 19th, 2026",91,0.2528,"1,000,000.0000","3,661,000,000.0000"
1,2,"May 19th, 2026","August 18th, 2026",91,0.2528,"875,000.0000","3,203,375,000.0000"
2,3,"August 18th, 2026","November 17th, 2026",91,0.2528,"750,000.0000","2,745,750,000.0000"
3,4,"November 17th, 2026","February 17th, 2027",92,0.2556,"625,000.0000","2,288,125,000.0000"
4,5,"February 17th, 2027","May 17th, 2027",89,0.2472,"500,000.0000","1,830,500,000.0000"
5,6,"May 17th, 2027","August 17th, 2027",92,0.2556,"375,000.0000","1,372,875,000.0000"
6,7,"August 17th, 2027","November 17th, 2027",92,0.2556,"250,000.0000","915,250,000.0000"
7,8,"November 17th, 2027","February 17th, 2028",92,0.2556,"125,000.0000","457,625,000.0000"


---
## 3. Curvas a Inception (2026-02-17)

### 3.1 ¿Qué le metemos a QuantLib?

**Curva IBR** — bootstrap OIS con:
- Depósitos (1D, 1M, 3M, 6M, 12M): tasas de BanRep (`banrep_series_value_v2`)
- Swaps (2Y, 5Y, 10Y): tasas par de DTCC (`ibr_2y`, `ibr_5y`, `ibr_10y`)
- Interpolación: `PiecewiseLogLinearDiscount`
- Day count: ACT/360

**Curva SOFR** — bootstrap OIS con:
- Depósitos (1M–12M) + Swaps (2Y–50Y): tasas par de Eris Futures (`sofr_swap_curve`)
- **OJO:** `market_marks.sofr` tiene **zero rates** — NO sirve para bootstrap. Se usan las tasas par.
- Settlement: T+2
- Interpolación: `PiecewiseLogLinearDiscount`

In [4]:
def build_cm(fecha_str: str) -> CurveManager:
    """Construye un CurveManager con las marcas de una fecha específica."""
    from datetime import datetime as dt
    # IBR: fetch_ibr_quotes devuelve {ibr_1d: [rate], ...} (formato correcto)
    ibr = loader.fetch_ibr_quotes(target_date=fecha_str)
    # SOFR: tasas par de sofr_swap_curve (NO market_marks.sofr)
    sofr_df = loader.fetch_sofr_curve(target_date=fecha_str)
    # FX: SET-ICAP intraday
    fx = loader.fetch_usdcop_spot(target_date=fecha_str)

    d = dt.strptime(fecha_str, '%Y-%m-%d')
    val_date = ql.Date(d.day, d.month, d.year)

    cm = CurveManager(valuation_date=val_date)
    cm.build_ibr_curve(ibr)
    cm.build_sofr_curve(sofr_df)
    cm.set_fx_spot(fx)
    return cm


print('=== Cargando marcas de inception: 2026-02-17 ===')
cm_inc = build_cm('2026-02-17')
print(f'FX spot:                  {cm_inc.fx_spot:.2f}')
print(f'IBR curve ref date:       {cm_inc.ibr_curve.referenceDate()}')
print(f'SOFR curve ref date:      {cm_inc.sofr_curve.referenceDate()}')

=== Cargando marcas de inception: 2026-02-17 ===
FX spot:                  3666.00
IBR curve ref date:       February 17th, 2026
SOFR curve ref date:      February 19th, 2026


### 3.2 Inputs de la curva IBR — nodos que entran al bootstrap

In [5]:
print('Nodos IBR (tasas par que entran al bootstrap):')
print(f'  {"Tenor":<12}  {"Tipo":<10}  {"Tasa (%)"}')
print('  ' + '-'*35)
for k, sq in cm_inc.ibr_quotes.items():
    tipo = 'Depósito' if k in ('ibr_1d','ibr_1m','ibr_3m','ibr_6m','ibr_12m') else 'OIS Swap'
    print(f'  {k:<12}  {tipo:<10}  {sq.value()*100:.4f}%')

Nodos IBR (tasas par que entran al bootstrap):
  Tenor         Tipo        Tasa (%)
  -----------------------------------
  ibr_1d        Depósito    9.6450%
  ibr_1m        Depósito    9.6640%
  ibr_3m        Depósito    10.3480%
  ibr_6m        Depósito    11.1250%
  ibr_12m       Depósito    12.0100%
  ibr_2y        OIS Swap    11.5200%
  ibr_5y        OIS Swap    10.9650%
  ibr_10y       OIS Swap    10.7000%


### 3.3 Inputs de la curva SOFR — nodos que entran al bootstrap

In [6]:
print('Nodos SOFR (tasas par que entran al bootstrap):')
print(f'  {"Tenor":>8}M   {"Tipo":<10}  {"Tasa (%)"}')
print('  ' + '-'*35)
for months, sq in cm_inc.sofr_quotes.items():
    tipo = 'Depósito' if months <= 12 else 'OIS Swap'
    label = f'{months}M' if months < 12 else f'{months//12}Y'
    print(f'  {label:>8}    {tipo:<10}  {sq.value()*100:.4f}%')

Nodos SOFR (tasas par que entran al bootstrap):
     TenorM   Tipo        Tasa (%)
  -----------------------------------
        1M    Depósito    3.6661%
        3M    Depósito    3.6627%
        6M    Depósito    3.6055%
        9M    Depósito    3.5152%
        1Y    Depósito    3.4315%
        1Y    OIS Swap    3.3007%
        2Y    OIS Swap    3.2578%
        3Y    OIS Swap    3.2484%
        4Y    OIS Swap    3.2876%
        5Y    OIS Swap    3.3459%
        6Y    OIS Swap    3.4108%
        7Y    OIS Swap    3.4761%
        8Y    OIS Swap    3.5371%
        9Y    OIS Swap    3.5952%
       10Y    OIS Swap    3.6511%
       12Y    OIS Swap    3.7557%
       15Y    OIS Swap    3.8791%
       20Y    OIS Swap    3.9892%
       25Y    OIS Swap    4.0116%
       30Y    OIS Swap    3.9919%
       40Y    OIS Swap    3.8995%
       50Y    OIS Swap    3.7999%


### 3.4 ¿Qué devuelve QuantLib? — Discount Factors y Zero Rates

Una vez bootstrappeado, la curva devuelve:
- **`discount(d)`** → factor de descuento $P(0,T)$ tal que $PV = CF \times P(0,T)$
- **`zeroRate(d, dc, Continuous)`** → tasa cero continua $r$ tal que $P(0,T) = e^{-rT}$
- **`forwardRate(s, e, dc, Simple)`** → tasa forward simple entre $s$ y $e$

In [7]:
key_dates = [
    ('3M',  ql.Date(17,5,2026)),
    ('6M',  ql.Date(17,8,2026)),
    ('9M',  ql.Date(17,11,2026)),
    ('12M', ql.Date(17,2,2027)),
    ('18M', ql.Date(17,8,2027)),
    ('24M', ql.Date(17,2,2028)),
]

rows = []
for label, d in key_dates:
    ibr_df  = cm_inc.ibr_handle.discount(d)
    ibr_zr  = cm_inc.ibr_handle.zeroRate(d, DC, ql.Continuous).rate()
    sofr_df = cm_inc.sofr_handle.discount(d)
    sofr_zr = cm_inc.sofr_handle.zeroRate(d, DC, ql.Continuous).rate()
    rows.append({
        'Tenor': label,
        'Fecha': str(d),
        'IBR DF': ibr_df,
        'IBR zero% (cont)': ibr_zr * 100,
        'SOFR DF': sofr_df,
        'SOFR zero% (cont)': sofr_zr * 100,
    })

df_curves = pd.DataFrame(rows)
df_curves

,Tenor,Fecha,IBR DF,IBR zero% (cont),SOFR DF,SOFR zero% (cont)
0,3M,"May 17th, 2026",0.9751,10.1989,0.9912,3.6464
1,6M,"August 17th, 2026",0.9471,10.8055,0.9824,3.5740
2,9M,"November 17th, 2026",0.9189,11.1487,0.9742,3.4707
3,12M,"February 17th, 2027",0.8916,11.3212,0.9665,3.3748
4,18M,"August 17th, 2027",0.8418,11.3510,0.9520,3.2548
5,24M,"February 17th, 2028",0.7942,11.3649,0.9373,3.2032


---
## 4. La Convención T+2 del SOFR — el problema y el fix

La curva SOFR tiene `settlement_days = 2`. Esto significa:

```
valuation_date = 2026-02-17
SOFR curve reference date = 2026-02-19  (T+2 hábil)
```

Cuando intentamos calcular el forward rate del **período 1** (2026-02-17 → 2026-05-17):
```python
sofr_handle.forwardRate(Date(17,2,2026), Date(17,5,2026), ...)
# → RuntimeError: negative time (-0.00555...)  ← ¡CRASH!
```

**¿Por qué?** QuantLib calcula el tiempo como `(start - referenceDate) / dayFraction`. Si `start < referenceDate`, el tiempo es negativo.

**Fix correcto:** Para la query de forward rate, usamos `max(period_start, sofr_ref_date)`.
- La `tau` (day count para el cashflow) sigue usando las fechas reales del período.
- Solo la consulta de forward rate usa la fecha clipeada.
- Esto es económicamente correcto: la fijación SOFR del primer cupón empieza en T+2 de la fecha de inicio.

In [8]:
sofr_ref = cm_inc.sofr_curve.referenceDate()
ibr_ref  = cm_inc.ibr_curve.referenceDate()

print(f'valuation_date (inception):  {T_START}')
print(f'SOFR curve referenceDate:    {sofr_ref}  ← T+2 = 2026-02-19')
print(f'IBR curve referenceDate:     {ibr_ref}')
print()
print(f'Período 1 start:  {DATES[0]}')
print(f'¿start < sofr_ref?  {DATES[0] < sofr_ref}  → necesitamos el clip')
print()
print(f'Sin clip → forwardRate({DATES[0]}, {DATES[1]}, ...) → RuntimeError: negative time')
print(f'Con clip → forwardRate({sofr_ref}, {DATES[1]}, ...)  → OK')
print(f'tau sigue siendo DC.yearFraction({DATES[0]}, {DATES[1]}) = {DC.yearFraction(DATES[0], DATES[1]):.6f}')

valuation_date (inception):  February 17th, 2026
SOFR curve referenceDate:    February 19th, 2026  ← T+2 = 2026-02-19
IBR curve referenceDate:     February 17th, 2026

Período 1 start:  February 17th, 2026
¿start < sofr_ref?  True  → necesitamos el clip

Sin clip → forwardRate(February 17th, 2026, May 19th, 2026, ...) → RuntimeError: negative time
Con clip → forwardRate(February 19th, 2026, May 19th, 2026, ...)  → OK
tau sigue siendo DC.yearFraction(February 17th, 2026, May 19th, 2026) = 0.252778


---
## 5. Valoración a Inception — Pata a Pata

### Fórmula general para cada período

$$CF_i = N_i \times (r^{fwd}_i + spread) \times \tau_i$$
$$PV_i = CF_i \times P(0, T_i)$$

donde:
- $N_i$ = nocional al inicio del período $i$
- $r^{fwd}_i$ = tasa forward simple de la curva entre $(s_i, e_i)$
- $\tau_i$ = fracción de año ACT/360 = días / 360
- $P(0, T_i)$ = discount factor al final del período

In [ ]:
def price_legs(cm, val_date: ql.Date, is_inception: bool):
    """
    Calcula período a período ambas patas del CCS.

    SOFR T+2 fix: para forwardRate() y discount() usa max(date, sofr_ref).
    La tau siempre usa las fechas reales del período.

    is_inception=True  → incluye notional exchange inicial en T_START
    is_inception=False → ese exchange ya se liquidó; solo retornos futuros
    """
    sofr_ref = cm.sofr_curve.referenceDate()
    ibr_ref  = cm.ibr_curve.referenceDate()
    spot     = cm.fx_spot

    usd_rows, cop_rows = [], []

    for i in range(1, len(DATES)):
        s, e = DATES[i-1], DATES[i]
        if e <= val_date:           # período ya vencido
            continue

        tau   = DC.yearFraction(s, e)
        n_usd = AMORT_USD[i-1]
        n_cop = AMORT_COP[i-1]

        # ── USD leg: SOFR − 22 bps ───────────────────────────────────────
        s_sofr   = sofr_ref if s < sofr_ref else s
        fwd_sofr = cm.sofr_handle.forwardRate(s_sofr, e, DC, ql.Simple).rate()
        allin    = fwd_sofr + SOFR_SPREAD
        cf_usd   = n_usd * allin * tau
        df_usd   = cm.sofr_handle.discount(e)
        pv_usd   = cf_usd * df_usd
        usd_rows.append({
            'P': i, 'start': str(s), 'end': str(e),
            'N_USD': n_usd,
            'SOFR_fwd%': round(fwd_sofr * 100, 4),
            'spread_bps': SOFR_SPREAD * 10_000,
            'allin%': round(allin * 100, 4),
            'tau': round(tau, 6),
            'CF_USD': round(cf_usd, 2),
            'DF_USD': round(df_usd, 6),
            'PV_USD': round(pv_usd, 2),
            'T+2 clip': 'si' if s < sofr_ref else '',
        })

        # ── COP leg: IBR flat ────────────────────────────────────────────
        s_ibr   = ibr_ref if s < ibr_ref else s
        fwd_ibr = cm.ibr_handle.forwardRate(s_ibr, e, DC, ql.Simple).rate()
        cf_cop  = n_cop * fwd_ibr * tau
        df_cop  = cm.ibr_handle.discount(e)
        pv_cop  = cf_cop * df_cop
        cop_rows.append({
            'P': i, 'start': str(s), 'end': str(e),
            'N_COP': n_cop,
            'IBR_fwd%': round(fwd_ibr * 100, 4),
            'tau': round(tau, 6),
            'CF_COP': round(cf_cop),
            'DF_COP': round(df_cop, 6),
            'PV_COP': round(pv_cop),
        })

    usd_df = pd.DataFrame(usd_rows)
    cop_df = pd.DataFrame(cop_rows)

    # ── Notional exchange PV ─────────────────────────────────────────────
    # Amortizable: cada trimestre se devuelve 12.5% del nocional
    # Convención: usd_nex = PV total de USD a pagar (salidas); cop_nex = PV total de COP a recibir (entradas)
    usd_nex, cop_nex = 0.0, 0.0
    for i in range(1, len(DATES)):
        e = DATES[i]
        if e <= val_date:
            continue
        usd_nex += USD_NOTIONAL * AMORT_FACTOR * cm.sofr_handle.discount(e)
        cop_nex += COP_NOTIONAL * AMORT_FACTOR * cm.ibr_handle.discount(e)

    if is_inception:
        # En inception: Empresa paga USD y recibe COP en T_START.
        # usd_nex acumula TODOS los USD que paga (salidas → positivo)
        # cop_nex acumula TODOS los COP que recibe (entradas → positivo)
        # SOFR T+2 fix: discount(T_START) falla si T_START < sofr_ref.
        # discount(sofr_ref) = 1.0 exactamente (es la referenceDate).
        sofr_t0 = sofr_ref if T_START < sofr_ref else T_START
        ibr_t0  = ibr_ref  if T_START < ibr_ref  else T_START
        usd_nex += USD_NOTIONAL * cm.sofr_handle.discount(sofr_t0)  # paga USD en T0 ≈ +1,000,000
        cop_nex += COP_NOTIONAL * cm.ibr_handle.discount(ibr_t0)    # recibe COP en T0 ≈ +3,661,000,000

    usd_leg_pv = usd_df['PV_USD'].sum()
    cop_leg_pv = cop_df['PV_COP'].sum()
    usd_total  = usd_leg_pv + usd_nex
    cop_total  = cop_leg_pv + cop_nex

    # NPV_COP = COP_total − USD_total × FX_spot
    # donde: cop_total = PV de todos los COP a recibir
    #        usd_total = PV de todos los USD a pagar
    npv_cop = cop_total - usd_total * spot
    npv_usd = npv_cop / spot

    return {
        'usd_df': usd_df, 'cop_df': cop_df,
        'usd_leg_pv': usd_leg_pv, 'cop_leg_pv': cop_leg_pv,
        'usd_nex': usd_nex, 'cop_nex': cop_nex,
        'usd_total': usd_total, 'cop_total': cop_total,
        'spot': spot, 'npv_cop': npv_cop, 'npv_usd': npv_usd,
    }


print('price_legs() definida.')


### 5.1 Pata USD — Empresa paga SOFR 3M − 22 bps

In [ ]:
res_inc = price_legs(cm_inc, val_date=T_START, is_inception=True)

print('=== PATA USD — inception 2026-02-17 ===')
print('Empresa PAGA: SOFR_fwd − 22bps × tau × N_USD, descontado con DF_USD')
print()
usd = res_inc['usd_df']
print(usd.to_string(index=False))
print()
print(f'  PV total pata USD:  {res_inc["usd_leg_pv"]:>14,.2f} USD')
print(f'  Notional exch PV:   {res_inc["usd_nex"]:>14,.2f} USD  (+USD pagado T0 + amort futuros a pagar)')
print(f'  USD TOTAL:          {res_inc["usd_total"]:>14,.2f} USD')

### 5.2 Pata COP — Bancolombia paga IBR trimestral flat

In [ ]:
print('=== PATA COP — inception 2026-02-17 ===')
print('Bancolombia PAGA: IBR_fwd × tau × N_COP, descontado con DF_COP')
print()
cop = res_inc['cop_df']
print(cop.to_string(index=False))
print()
print(f'  PV total pata COP:  {res_inc["cop_leg_pv"]:>20,.0f} COP')
print(f'  Notional exch PV:   {res_inc["cop_nex"]:>20,.0f} COP  (+COP recibido T0 + amort futuros a recibir)')
print(f'  COP TOTAL:          {res_inc["cop_total"]:>20,.0f} COP')

### 5.3 NPV a Inception

$$NPV_{COP} = COP_{total} - USD_{total} \times FX_{spot}$$

En un swap at-market el NPV debería ser **~0** en inception. Cualquier residual refleja bid-offer o diferencias de interpolación de curva.

In [12]:
d = res_inc

print('=' * 60)
print('NPV INCEPTION — perspectiva EMPRESA ABC SA')
print('=' * 60)
print()
print(f'  FX spot (inception):      {d["spot"]:>14,.2f}')
print()
print(f'  COP leg PV:               {d["cop_leg_pv"]:>18,.0f} COP')
print(f'  COP notional exch:        {d["cop_nex"]:>18,.0f} COP')
print(f'  ─────────────────────────────────────────────')
print(f'  COP TOTAL:                {d["cop_total"]:>18,.0f} COP')
print()
print(f'  USD leg PV:               {d["usd_leg_pv"]:>14,.2f} USD')
print(f'  USD notional exch:        {d["usd_nex"]:>14,.2f} USD')
print(f'  ─────────────────────────────────────────────')
print(f'  USD TOTAL:                {d["usd_total"]:>14,.2f} USD')
print()
print(f'  NPV_COP = COP_total − USD_total × spot')
print(f'          = {d["cop_total"]:,.0f}')
print(f'            − {d["usd_total"]:,.2f} × {d["spot"]:.2f}')
print(f'          = {d["cop_total"]:,.0f} − {d["usd_total"]*d["spot"]:,.0f}')
print()
print(f'  NPV (COP):  {d["npv_cop"]:>18,.0f}')
print(f'  NPV (USD):  {d["npv_usd"]:>18,.2f}')

NPV INCEPTION — perspectiva EMPRESA ABC SA

  FX spot (inception):            3,666.00

  COP leg PV:                      433,703,281 COP
  COP notional exch:             6,888,296,720 COP
  ─────────────────────────────────────────────
  COP TOTAL:                     7,322,000,001 COP

  USD leg PV:                    34,349.35 USD
  USD notional exch:            -36,583.00 USD
  ─────────────────────────────────────────────
  USD TOTAL:                     -2,233.65 USD

  NPV_COP = COP_total − USD_total × spot
          = 7,322,000,001
            − -2,233.65 × 3666.00
          = 7,322,000,001 − -8,188,567

  NPV (COP):       7,330,188,568
  NPV (USD):        1,999,505.88


---
## 6. Curvas de Hoy (2026-03-03) — Mid-life Pricing

El swap lleva 14 días corriendo. ¿Qué cambió?
- **FX spot**: posiblemente se movió desde 3,661
- **IBR curve**: puede haberse desplazado
- **SOFR curve**: puede haberse desplazado
- **Notional exchange inicial**: ya se liquidó el 17-feb → `is_inception=False`
- **Período 1** (Feb17→May17): está en curso, período aún no vencido

In [13]:
print('=== Cargando marcas de hoy: 2026-03-03 ===')
cm_hoy = build_cm('2026-03-03')
print(f'FX spot:                  {cm_hoy.fx_spot:.2f}')
print(f'SOFR curve ref date:      {cm_hoy.sofr_curve.referenceDate()}')
print(f'IBR  curve ref date:      {cm_hoy.ibr_curve.referenceDate()}')
print()
print(f'FX move:  {FX_INCEPTION:.2f} → {cm_hoy.fx_spot:.2f}  ({(cm_hoy.fx_spot/FX_INCEPTION-1)*100:+.2f}%)')

=== Cargando marcas de hoy: 2026-03-03 ===
FX spot:                  3803.00
SOFR curve ref date:      March 5th, 2026
IBR  curve ref date:      March 3rd, 2026

FX move:  3661.00 → 3803.00  (+3.88%)


### 6.1 Comparación de curvas: inception vs hoy

In [14]:
check_dates = [
    ('3M',  DATES[1]),
    ('6M',  DATES[2]),
    ('12M', DATES[4]),
    ('18M', DATES[6]),
    ('24M', DATES[8]),
]

rows = []
for label, d in check_dates:
    ibr_i  = cm_inc.ibr_handle.zeroRate(d, DC, ql.Continuous).rate()
    ibr_h  = cm_hoy.ibr_handle.zeroRate(d, DC, ql.Continuous).rate()
    sofr_i = cm_inc.sofr_handle.zeroRate(d, DC, ql.Continuous).rate()
    sofr_h = cm_hoy.sofr_handle.zeroRate(d, DC, ql.Continuous).rate()
    rows.append({
        'Tenor': label,
        'Fecha': str(d),
        'IBR inc%': round(ibr_i * 100, 4),
        'IBR hoy%': round(ibr_h * 100, 4),
        'IBR Δbps': round((ibr_h - ibr_i) * 10000, 1),
        'SOFR inc%': round(sofr_i * 100, 4),
        'SOFR hoy%': round(sofr_h * 100, 4),
        'SOFR Δbps': round((sofr_h - sofr_i) * 10000, 1),
    })

pd.DataFrame(rows)

,Tenor,Fecha,IBR inc%,IBR hoy%,IBR Δbps,SOFR inc%,SOFR hoy%,SOFR Δbps
0,3M,"May 19th, 2026",10.1292,10.3749,24.6000,3.6475,3.6575,1.0000
1,6M,"August 18th, 2026",10.7327,11.0312,29.8000,3.5812,3.6018,2.1000
2,12M,"February 17th, 2027",11.2991,11.7297,43.1000,3.3869,3.4526,6.6000
3,18M,"August 17th, 2027",11.3493,11.9739,62.5000,3.2631,3.3402,7.7000
4,24M,"February 17th, 2028",11.3637,12.0956,73.2000,3.2062,3.2714,6.5000


### 6.2 Pata USD — hoy

In [15]:
VAL_HOY = ql.Date(3, 3, 2026)
res_hoy = price_legs(cm_hoy, val_date=VAL_HOY, is_inception=False)

print('=== PATA USD — hoy 2026-03-03  (solo períodos futuros) ===')
print(res_hoy['usd_df'].to_string(index=False))
print()
print(f'  PV total pata USD:  {res_hoy["usd_leg_pv"]:>14,.2f} USD')
print(f'  Notional exch PV:   {res_hoy["usd_nex"]:>14,.2f} USD  (solo retornos futuros)')
print(f'  USD TOTAL:          {res_hoy["usd_total"]:>14,.2f} USD')

=== PATA USD — hoy 2026-03-03  (solo períodos futuros) ===
 P               start                 end          N_USD  SOFR_fwd%  spread_bps  allin%    tau     CF_USD  DF_USD     PV_USD T+2 clip
 1 February 17th, 2026      May 19th, 2026 1,000,000.0000     3.6715    -22.0000  3.4515 0.2528 8,724.6800  0.9924 8,658.4500       si
 2      May 19th, 2026   August 18th, 2026   875,000.0000     3.5719    -22.0000  3.3519 0.2528 7,413.8200  0.9835 7,291.7100         
 3   August 18th, 2026 November 17th, 2026   750,000.0000     3.4062    -22.0000  3.1862 0.2528 6,040.5600  0.9751 5,890.3400         
 4 November 17th, 2026 February 17th, 2027   625,000.0000     3.2572    -22.0000  3.0372 0.2556 4,851.0300  0.9671 4,691.3500         
 5 February 17th, 2027      May 17th, 2027   500,000.0000     3.1446    -22.0000  2.9246 0.2472 3,615.1500  0.9596 3,469.1800         
 6      May 17th, 2027   August 17th, 2027   375,000.0000     3.1269    -22.0000  2.9069 0.2556 2,785.7900  0.9520 2,652.1200      

### 6.3 Pata COP — hoy

In [16]:
print('=== PATA COP — hoy 2026-03-03  (solo períodos futuros) ===')
print(res_hoy['cop_df'].to_string(index=False))
print()
print(f'  PV total pata COP:  {res_hoy["cop_leg_pv"]:>20,.0f} COP')
print(f'  Notional exch PV:   {res_hoy["cop_nex"]:>20,.0f} COP  (solo retornos futuros)')
print(f'  COP TOTAL:          {res_hoy["cop_total"]:>20,.0f} COP')

=== PATA COP — hoy 2026-03-03  (solo períodos futuros) ===
 P               start                 end              N_COP  IBR_fwd%    tau   CF_COP  DF_COP   PV_COP
 1 February 17th, 2026      May 19th, 2026 3,661,000,000.0000   10.4909 0.2528 97084813  0.9781 94954150
 2      May 19th, 2026   August 18th, 2026 3,203,375,000.0000   11.7579 0.2528 95208764  0.9498 90431522
 3   August 18th, 2026 November 17th, 2026 2,745,750,000.0000   12.4864 0.2528 86663596  0.9208 79796518
 4 November 17th, 2026 February 17th, 2027 2,288,125,000.0000   12.6475 0.2556 73955478  0.8919 65963341
 5 February 17th, 2027      May 17th, 2027 1,830,500,000.0000   12.6409 0.2472 57205278  0.8649 49477065
 6      May 17th, 2027   August 17th, 2027 1,372,875,000.0000   12.6475 0.2556 44373287  0.8378 37177013
 7   August 17th, 2027 November 17th, 2027   915,250,000.0000   12.6475 0.2556 29582191  0.8116 24008680
 8 November 17th, 2027 February 17th, 2028   457,625,000.0000   12.6475 0.2556 14791096  0.7862 11628

### 6.4 NPV hoy

In [17]:
d = res_hoy

print('=' * 60)
print('NPV HOY (2026-03-03) — perspectiva EMPRESA ABC SA')
print('=' * 60)
print()
print(f'  FX spot (hoy):            {d["spot"]:>14,.2f}')
print()
print(f'  COP leg PV:               {d["cop_leg_pv"]:>18,.0f} COP')
print(f'  COP notional exch:        {d["cop_nex"]:>18,.0f} COP')
print(f'  COP TOTAL:                {d["cop_total"]:>18,.0f} COP')
print()
print(f'  USD leg PV:               {d["usd_leg_pv"]:>14,.2f} USD')
print(f'  USD notional exch:        {d["usd_nex"]:>14,.2f} USD')
print(f'  USD TOTAL:                {d["usd_total"]:>14,.2f} USD')
print()
print(f'  NPV (COP):  {d["npv_cop"]:>18,.0f}')
print(f'  NPV (USD):  {d["npv_usd"]:>18,.2f}')

NPV HOY (2026-03-03) — perspectiva EMPRESA ABC SA

  FX spot (hoy):                  3,803.00

  COP leg PV:                      453,436,780 COP
  COP notional exch:             3,222,171,550 COP
  COP TOTAL:                     3,675,608,330 COP

  USD leg PV:                    35,241.95 USD
  USD notional exch:            963,940.15 USD
  USD TOTAL:                    999,182.10 USD

  NPV (COP):        -124,281,195
  NPV (USD):          -32,679.78


---
## 7. Resumen: Inception vs Hoy

In [18]:
i, h = res_inc, res_hoy
W = 18

print('=' * 68)
print('RESUMEN — EMPRESA ABC SA')
print('=' * 68)
print(f'  {"":28}  {"INCEPTION 17-feb":>{W}}  {"HOY 03-mar":>{W}}  {"DELTA":>12}')
print(f'  {"-"*70}')
print(f'  {"FX spot":28}  {FX_INCEPTION:>{W},.2f}  {cm_hoy.fx_spot:>{W},.2f}  {cm_hoy.fx_spot-FX_INCEPTION:>+12,.2f}')
print(f'  {"-"*70}')
print(f'  {"USD leg PV (USD)":28}  {i["usd_leg_pv"]:>{W},.2f}  {h["usd_leg_pv"]:>{W},.2f}')
print(f'  {"USD notional exch (USD)":28}  {i["usd_nex"]:>{W},.2f}  {h["usd_nex"]:>{W},.2f}')
print(f'  {"COP leg PV (COP)":28}  {i["cop_leg_pv"]:>{W},.0f}  {h["cop_leg_pv"]:>{W},.0f}')
print(f'  {"COP notional exch (COP)":28}  {i["cop_nex"]:>{W},.0f}  {h["cop_nex"]:>{W},.0f}')
print(f'  {"-"*70}')
print(f'  {"NPV (COP)":28}  {i["npv_cop"]:>{W},.0f}  {h["npv_cop"]:>{W},.0f}  {h["npv_cop"]-i["npv_cop"]:>+12,.0f}')
print(f'  {"NPV (USD)":28}  {i["npv_usd"]:>{W},.0f}  {h["npv_usd"]:>{W},.0f}  {h["npv_usd"]-i["npv_usd"]:>+12,.0f}')

RESUMEN — EMPRESA ABC SA
                                  INCEPTION 17-feb          HOY 03-mar         DELTA
  ----------------------------------------------------------------------
  FX spot                                 3,661.00            3,803.00       +142.00
  ----------------------------------------------------------------------
  USD leg PV (USD)                       34,349.35           35,241.95
  USD notional exch (USD)               -36,583.00          963,940.15
  COP leg PV (COP)                     433,703,281         453,436,780
  COP notional exch (COP)            6,888,296,720       3,222,171,550
  ----------------------------------------------------------------------
  NPV (COP)                          7,330,188,568        -124,281,195  -7,454,469,763
  NPV (USD)                              1,999,506             -32,680    -2,032,186


---
## 8. Descomposición del P&L: FX vs Tasas

### ¿Qué mueve el NPV de este swap?

**Efecto FX:** La exposición dominante en un CCS es el notional exchange.
- Empresa paga USD y recibe COP al FX inception (3,661).
- Al vencimiento de cada trimestre devuelve 12.5% del nocional.
- Si el COP se deprecia (USD/COP sube), los USD que recibirá valen **más** en COP.
- El notional exchange final en USD se descuenta con SOFR y se convierte al FX **de hoy** → eso es P&L FX.

**Efecto tasas:**
- IBR sube → los COP futuros (que recibe) valen más → NPV sube
- SOFR baja → los USD futuros (que paga) valen menos → NPV sube

In [19]:
# ── P&L FX: repricing COP total con FX inception ──────────────────────────
# Si no hubiera cambiado el FX, el NPV_COP hoy con FX = 3661:
npv_hoy_fx_frozen = res_hoy['cop_total'] - res_hoy['usd_total'] * FX_INCEPTION

# P&L FX = NPV_hoy - NPV_hoy_fx_frozen
pnl_fx    = res_hoy['npv_cop'] - npv_hoy_fx_frozen
# P&L rates = NPV_hoy_fx_frozen - NPV_inception
pnl_rates = npv_hoy_fx_frozen - res_inc['npv_cop']
pnl_total = res_hoy['npv_cop'] - res_inc['npv_cop']

print('=== Descomposición P&L (14 días) ===')
print()
print(f'  NPV inception (17-feb):    {res_inc["npv_cop"]:>14,.0f} COP')
print(f'  NPV hoy (03-mar):          {res_hoy["npv_cop"]:>14,.0f} COP')
print(f'  P&L total:                 {pnl_total:>+14,.0f} COP')
print()
print(f'  Descomposición:')
print(f'    P&L FX    (FX: {FX_INCEPTION:.0f} → {cm_hoy.fx_spot:.0f}):  {pnl_fx:>+14,.0f} COP')
print(f'    P&L Rates (curvas IBR/SOFR):    {pnl_rates:>+14,.0f} COP')
print(f'    Check suma:                     {pnl_fx+pnl_rates:>+14,.0f} COP')

=== Descomposición P&L (14 días) ===

  NPV inception (17-feb):     7,330,188,568 COP
  NPV hoy (03-mar):            -124,281,195 COP
  P&L total:                 -7,454,469,763 COP

  Descomposición:
    P&L FX    (FX: 3661 → 3803):    -141,883,858 COP
    P&L Rates (curvas IBR/SOFR):    -7,312,585,905 COP
    Check suma:                     -7,454,469,763 COP


---
## 9. DV01 — Sensibilidad a +1 bp

DV01 = cambio en NPV (COP) cuando **toda** la curva se desplaza +1 bp.

In [20]:
base_npv = res_hoy['npv_cop']

# DV01 IBR
cm_hoy.bump_ibr(1)
npv_ibr_bumped = price_legs(cm_hoy, val_date=VAL_HOY, is_inception=False)['npv_cop']
dv01_ibr = npv_ibr_bumped - base_npv
cm_hoy.bump_ibr(-1)

# DV01 SOFR
cm_hoy.bump_sofr(1)
npv_sofr_bumped = price_legs(cm_hoy, val_date=VAL_HOY, is_inception=False)['npv_cop']
dv01_sofr = npv_sofr_bumped - base_npv
cm_hoy.bump_sofr(-1)

print('=== DV01 (hoy, +1bp paralelo) ===')
print()
print(f'  Base NPV (COP):     {base_npv:>14,.0f}')
print()
print(f'  DV01 IBR  (COP):    {dv01_ibr:>+14,.0f}  ← IBR sube 1bp → recibe más → NPV sube')
print(f'  DV01 SOFR (COP):    {dv01_sofr:>+14,.0f}  ← SOFR sube 1bp → paga más  → NPV baja')
print(f'  DV01 neto (COP):    {dv01_ibr+dv01_sofr:>+14,.0f}')
print()
print(f'  DV01 IBR  (USD):    {dv01_ibr/cm_hoy.fx_spot:>+10,.0f}')
print(f'  DV01 SOFR (USD):    {dv01_sofr/cm_hoy.fx_spot:>+10,.0f}')

=== DV01 (hoy, +1bp paralelo) ===

  Base NPV (COP):       -124,281,195

  DV01 IBR  (COP):           +13,595  ← IBR sube 1bp → recibe más → NPV sube
  DV01 SOFR (COP):           -17,267  ← SOFR sube 1bp → paga más  → NPV baja
  DV01 neto (COP):            -3,671

  DV01 IBR  (USD):            +4
  DV01 SOFR (USD):            -5
